---
---
# **Tutorial 4:** *Agentic Frameworks — Google ADK & MOYA*
### *One personal-shopper agent, built twice, so you can see what a framework actually buys you*
---
---

### QUESTIONS FOR TODAY
> 1. *Past tutorials described the components of an agentic system. So what's left for a "framework" to do?*
> 2. *If an agent can browse a website and click "add to cart" on its own — should it also be allowed to click "pay"?*

### TODAY'S SCENARIO 🛒
> You hand an assistant a scribbled note: *"grab a blue top and some jeans for the weekend."* It should read the note, search an actual online store, add the right items to a cart, keep a running total against your budget — **and then stop, cart in hand, and wait for you to say go.**

> We'll build this **AI Personal Shopper** once in **Google ADK**, once in **MOYA** — same tools, same task, so the only thing that changes between the two halves of today is *how much of the orchestration the framework gives you for free.*

### OUR ROADMAP
| Section | What we're building |
|---|---|
| **Step 0** | Ollama check + installs (both frameworks are model-agnostic) |
| **Section 1 : Google ADK** | One `LlmAgent`, three tools, a real live product search |
| **Section 2 : MOYA** | Same tools, same agent — wrapped in an explicit, inspectable **Pipeline** |
| **Section 2B : MOYA** | A smaller use case: two agents, and a **classifier** that picks between them |
| **Key Takeaways** | Why the agent structurally *cannot* pay, even if you asked it to |

### THE ONE IDEA
> **A framework doesn't replace the tool-calling loop or MCP — it sits one layer up, running that loop for you and giving you a place to put the next step.** Today's real safety lesson: the agent stops before checkout not because we told it to be careful, but because **no `checkout` tool exists**. A capability an agent doesn't have is a promise that can't be broken.

> *Official docs, for reference: 

- [google/adk-python](https://github.com/google/adk-python) ([ADK docs](https://google.github.io/adk-docs/))

- [montycloud/moya](https://github.com/montycloud/moya) (paper: [arXiv:2501.08243](https://arxiv.org/abs/2501.08243)).

---
# **Step 0: Setting Up**
---

```bash
ollama serve                 # in its own terminal
ollama pull qwen3.5:4b       # one-time, ~2.5 GB
```

> ⚠️ A version note: MOYA's **PyPI package lags its GitHub repo**. `pip install moya-ai` currently gives you a version whose `OllamaAgent` can chat but **cannot call tools** (it hits Ollama's older `/api/generate` endpoint). The tool-calling `OllamaAgent` we need today — registry, classifier, `Pipeline` — only exists on the **`main` branch**, so we install straight from GitHub.
> *Worth internalising as a habit, not a one-off: for a framework this young, check whether `pip install <name>` is actually the version the docs describe.*

> 🌐 About the "real store" we'll shop against: [automationexercise.com](https://automationexercise.com) is a public demo storefront built specifically for automation practice — it even documents its own [`GET /api/productsList`](https://automationexercise.com/api_list) endpoint. We hit that real, live, public API today. If your room's wifi has a bad minute, every tool below falls back to a small built-in catalog automatically — the demo never actually breaks mid-session.

In [ ]:
# ---------------------------------------------------------
# 1. INSTALL BOTH FRAMEWORKS (no API keys — everything is local Ollama)
# ---------------------------------------------------------
%pip install -q google-adk litellm nest_asyncio requests
%pip install -q "git+https://github.com/montycloud/moya.git"

# ---------------------------------------------------------
# 2. CONFIRM OLLAMA IS UP AND THE MODEL IS PULLED
# ---------------------------------------------------------
import requests

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3.5:4b"

try:
    tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    names = [m["name"] for m in tags.get("models", [])]
    print("Ollama is up. Models available:", names)
    if not any(MODEL_NAME in n for n in names):
        print(f"⚠️  {MODEL_NAME} not found — run: ollama pull {MODEL_NAME}")
except Exception as e:
    print("⚠️  Can't reach Ollama. Run `ollama serve` in a terminal first.\n", e)

### The three tools, shared by both frameworks

We build these **once** — a framework's whole job is to wrap tools you already know how to write.

In [ ]:
# ---------------------------------------------------------
# STEP 1 — THE THREE TOOLS
#
# search_product : hits the REAL automationexercise.com catalog API.
#                   Falls back to a small local list if the network hiccups —
#                   the demo never depends on the venue's wifi being perfect.
# add_to_cart     : simulated locally (a plain Python list). In production
#                   this would call the store's authenticated cart API —
#                   but we don't want a live class depending on a real
#                   session/cookie flow that could break mid-demo.
# check_budget    : pure local arithmetic.
#
# Notice what's MISSING: there is no checkout / pay / place_order tool.
# That's not an oversight — it's the whole safety story for today.
# ---------------------------------------------------------
FALLBACK_CATALOG = [
    {"name": "Blue Top", "price": 500},
    {"name": "Men Tshirt", "price": 400},
    {"name": "Soft Stretch Jeans", "price": 799},
    {"name": "Winter Top", "price": 600},
]
CART = []
BUDGET = 1850  # rupees


def _get_catalog():
    try:
        r = requests.get("https://automationexercise.com/api/productsList", timeout=5)
        data = r.json()
        return [{"name": p["name"], "price": int(p["price"].replace("Rs. ", ""))}
                for p in data["products"]]
    except Exception:
        return FALLBACK_CATALOG


def search_product(query: str) -> str:
    """Search the store's live product catalog for an item.

    Args:
        query: what to look for, e.g. "jeans" or "blue top"
    """
    catalog = _get_catalog()
    matches = [p for p in catalog if query.lower() in p["name"].lower()]
    if not matches:
        return f"No match found for '{query}'."
    best = min(matches, key=lambda p: p["price"])
    return f"{best['name']} — Rs. {best['price']}"


def add_to_cart(item_name: str, price: float) -> str:
    """Add an item to the shopping cart.

    Args:
        item_name: exact product name to add
        price: price in rupees
    """
    price_int = int(price)
    CART.append((item_name, price_int))
    total = sum(p for _, p in CART)
    return f"Added {item_name} (Rs. {price_int}). Cart total so far: Rs. {total}"


def check_budget() -> str:
    """Check the current cart total against the shopper's budget."""
    total = sum(p for _, p in CART)
    if total <= BUDGET:
        return f"Within budget: Rs. {total} of Rs. {BUDGET}."
    return f"Over budget by Rs. {total - BUDGET} (cart Rs. {total}, budget Rs. {BUDGET})."

# quick sanity check — a REAL network call, not a mock
print(search_product("jeans"))

---
# **Section 1 : Google ADK**
---

Straight from [Google's own description](https://github.com/google/adk-python): ADK is *"a flexible and modular framework for developing and deploying AI agents… optimized for Gemini and the Google ecosystem, but model-agnostic."* That last part is why we can point it at Ollama and lose nothing.

The idea:
| ADK concept | You already know it as… |
|---|---|
| `LlmAgent` | the model + a system prompt, wrapped up |
| a Python function in `tools=[...]` | exactly the tools you wrote by hand, docstring and all |
| `Runner` + `InMemorySessionService` | the loop you wrote yourself — *ask → model requests a tool → you run it → send result back* |

No `sub_agents` today — one agent is genuinely enough for this task. Watch what the framework buys you even in the single-agent case: schema generation, the session, and the loop that keeps calling tools **until the model itself decides it's done**.

In [ ]:
# ---------------------------------------------------------
# STEP 2 — THE MODEL, POINTED AT OLLAMA
#
# LiteLLM is ADK's bridge to non-Gemini models. "ollama_chat/" tells it
# which local model to drive over Ollama's /api/chat endpoint.
# ---------------------------------------------------------
import asyncio
import nest_asyncio
import litellm

nest_asyncio.apply()  # lets `await` work inside a notebook cell

# Increase request timeout for local Ollama tool-calling loops (in seconds)
litellm.request_timeout = 120

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

MODEL = LiteLlm(model=f"ollama_chat/{MODEL_NAME}", api_base=OLLAMA_URL)
print("Model wired to local Ollama:", MODEL_NAME)

In [ ]:
# ---------------------------------------------------------
# STEP 3 — ONE AGENT, THREE TOOLS
#
# The instruction does the sequencing: "for each item... then check_budget...
# then STOP." ADK's Runner will keep calling the model — and the model will
# keep calling tools — until a response comes back with no tool call in it.
# That loop is the framework's whole contribution here.
# ---------------------------------------------------------
shopper = LlmAgent(
    name="personal_shopper",
    model=MODEL,
    description="Shops a to-do list against the store catalog and stops before payment.",
    instruction=(
        "You are a careful personal shopping assistant. Read the shopping note. "
        "For each distinct item mentioned, call search_product to find it, then "
        "add_to_cart with the exact name and price returned. Once every item is "
        "added, call check_budget. Then summarise the cart and STOP — you have no "
        "tool to pay or check out, and must never claim to have done so."
    ),
    tools=[search_product, add_to_cart, check_budget],
)

async def ask(agent, prompt, session_id):
    runner = InMemoryRunner(agent=agent, app_name="shopper_app")
    await runner.session_service.create_session(
        app_name="shopper_app", user_id="you", session_id=session_id
    )
    content = types.Content(role="user", parts=[types.Part(text=prompt)])
    
    final_text = None
    
    # Iterate through all events to let ADK close its context generator cleanly
    async for event in runner.run_async(user_id="you", session_id=session_id, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_text = event.content.parts[0].text

    return final_text or "No response generated."

note = "I need a blue top and some jeans for the weekend."
print("----------------------------------------------------------\n")
print("-----------------------------Shopping note:", note, "\n")
print("----------------------------------------------------------\n")
print(await ask(shopper, note, "adk-run-1"))
print("----------------------------------------------------------\n")
print("-----------------------------Cart contents:", CART, "\n")
print("----------------------------------------------------------\n")

Onward — same tools, same task, MOYA's version.

---
# **Section 2 : MOYA**
---

[MOYA](https://github.com/montycloud/moya) — **M**eta **O**rchestrator of **Y**our **A**gents — comes out of MontyCloud's CloudOps platform. Same three tools today, but MOYA gives us one more piece worth seeing: a **`Pipeline`** — a declarative, inspectable sequence of steps, instead of one agent silently deciding everything internally.

| MOYA concept | Role |
|---|---|
| `OllamaAgent` | one agent, one model — same idea as `LlmAgent` |
| `ToolRegistry` + `Tool` | MOYA's version of the schema `@tool()` auto-generated |
| `AgentStep` | runs an agent and feeds its output into the next step |
| `FunctionStep` | a plain Python function as a pipeline step — **no LLM call at all** |
| `Pipeline` | chains steps in a fixed, visible order |

In [ ]:
# ---------------------------------------------------------
# STEP 1 — THE SAME THREE TOOLS, WRAPPED MOYA-STYLE
# ---------------------------------------------------------
from moya.agents.ollama_agent import OllamaAgent, OllamaAgentConfig
from moya.tools.tool import Tool
from moya.tools.tool_registry import ToolRegistry
from moya.flows.pipeline import Pipeline, FlowContext
from moya.flows.steps import AgentStep, FunctionStep

shopper_tools = ToolRegistry()
shopper_tools.register_tool(Tool(name="search_product", function=search_product))
shopper_tools.register_tool(Tool(name="add_to_cart", function=add_to_cart))
shopper_tools.register_tool(Tool(name="check_budget", function=check_budget, parameters={}))

shopper_m = OllamaAgent(OllamaAgentConfig(
    agent_name="personal_shopper",
    agent_type="ChatAgent",
    description="Shops a to-do list against the store catalog and stops before payment.",
    system_prompt=(
        "You are a careful personal shopping assistant. Read the shopping note. "
        "For each distinct item mentioned, call search_product to find it, then "
        "add_to_cart with the exact name and price returned. Once every item is "
        "added, call check_budget. Then summarise the cart and STOP — you have no "
        "tool to pay or check out, and must never claim to have done so."
    ),
    tool_registry=shopper_tools,
    model_name=MODEL_NAME,
    base_url=OLLAMA_URL,
    # More items on the list = more tool round-trips. 
    # MAX_TURNS cap — raise it to fit your longest expected list.
    max_iterations=15,
))

In [ ]:
# ---------------------------------------------------------
# STEP 2 — A TWO-STEP PIPELINE: AGENT, THEN A PLAIN FUNCTION
#
# AgentStep    -> the same shopping loop as the ADK version, inside one agent
# FunctionStep -> zero LLM calls. Guaranteed to run, guaranteed to say "stop".
# ---------------------------------------------------------
def present_and_stop(ctx: FlowContext) -> FlowContext:
    total = sum(int(p) for _, p in CART)
    lines = "\n".join(f"  - {n}: Rs. {p}" for n, p in CART) if CART else "  - No items in cart"
    
    agent_output = ctx.output if ctx.output else "Agent run complete."
    ctx.output = (
        f"{agent_output}\n\n"
        f"--- CART SUMMARY ---\n{lines}\nTotal: Rs. {total}\n"
        f"Stopping here — no checkout tool exists. A human must approve payment."
    )
    return ctx

pipeline = Pipeline([
    AgentStep(shopper_m),
    FunctionStep(present_and_stop),
])

CART.clear()  # fresh cart for this run
result = pipeline.run(thread_id="moya-run-1", message=note)
print("----------------------------------------------------------\n")
print(result)
print("----------------------------------------------------------\n")

---
# **Section 2B : A second, smaller MOYA use case**
### *Routing — the "MO" in MOYA*
---

The shopper used one agent. But MOYA is **M**eta **O**rchestrator of **Y**our **A**gents — and we haven't yet seen the orchestrating.

Here's the smaller scenario: a customer types into the store's chat box. Sometimes it's *"does this go with black jeans?"* — a styling question. Sometimes it's *"can I send this back?"* — a policy question. **One agent with one giant system prompt could try to cover both. Two narrow agents and a router is the cleaner shape**, and it's the same shape whether you have 2 agents or 50.

| Piece | Role |
|---|---|
| `AgentRegistry` | the list of who's available, and what each one is for |
| `Classifier` | looks at the message, returns *one agent name* |
| `MultiAgentOrchestrator` | asks the classifier, then hands the message to the winner |

No tools, no pipeline — this is about 30 lines. Notice that **the classifier routes on the `description=` field** we've been filling in all along. Descriptions aren't documentation here; they're the routing table.

In [ ]:
# ---------------------------------------------------------
# STEP 1 — TWO SMALL AGENTS + A REGISTRY
#
# Same OllamaAgent class as the shopper, minus the tools.
# The `description` is what the classifier will read to decide.
# ---------------------------------------------------------
from moya.registry.agent_registry import AgentRegistry
from moya.classifiers.llm_classifier import LLMClassifier
from moya.orchestrators.multi_agent_orchestrator import MultiAgentOrchestrator


def make_agent(name, description, system_prompt):
    return OllamaAgent(OllamaAgentConfig(
        agent_name=name,
        agent_type="ChatAgent",
        description=description,
        system_prompt=system_prompt,
        model_name=MODEL_NAME,
        base_url=OLLAMA_URL,
    ))


style_agent = make_agent(
    "style_advisor",
    "Advises on what to wear: sizing, colours, fabrics and outfit combinations.",
    "You are a friendly fashion advisor. Answer in two sentences, maximum.",
)

returns_agent = make_agent(
    "returns_desk",
    "Handles orders, delivery, refunds and the store returns policy.",
    "You are the store returns desk. Policy: returns accepted within 30 days "
    "with a receipt, unworn. Answer in two sentences, maximum.",
)

registry = AgentRegistry()
registry.register_agent(style_agent)
registry.register_agent(returns_agent)

print("Registered:", [a.name for a in registry.list_agents()])

In [ ]:
# ---------------------------------------------------------
# STEP 2 — THE CLASSIFIER, AND ONE HONEST PATCH
#
# MOYA's LLMClassifier asks a model for an agent name and then requires an
# EXACT string match. qwen3.5 is a thinking model — it happily replies with a
# paragraph of reasoning, the exact match fails, and every message silently
# lands on the default agent. You'd never see it; you'd just think routing
# doesn't work.
#
# Classifier is an interface, so we override the one method. Note we take the
# LAST agent name mentioned: a thinking model reasons about the options first
# and states its conclusion at the end.
# ---------------------------------------------------------
class TolerantLLMClassifier(LLMClassifier):
    """LLMClassifier that tolerates a chatty model."""

    def classify(self, message, thread_id=None, available_agents=None):
        if not available_agents:
            return self.default_agent
        menu = "\n".join(f"- {a.name}: {a.description}" for a in available_agents)
        reply = self.llm_agent.handle_message(
            f"Available agents:\n{menu}\n\nUser message: {message}\n\n"
            f"Reply with exactly one agent name and nothing else.",
            thread_id=thread_id,
        )
        hits = [(reply.rfind(a.name), a.name) for a in available_agents if a.name in reply]
        return max(hits)[1] if hits else self.default_agent


classifier = TolerantLLMClassifier(
    make_agent("router", "Picks the right agent.",
               "You are a router. Reply with exactly one agent name, nothing else."),
    default_agent="style_advisor",
)

orchestrator = MultiAgentOrchestrator(agent_registry=registry, classifier=classifier)

# ---------------------------------------------------------
# STEP 3 — TWO MESSAGES, TWO DESTINATIONS
# The [agent_name] prefix in the output is the orchestrator telling us who answered.
# ---------------------------------------------------------
for msg in [
    "Does a blue top actually go with black jeans, or is that a bit much?",
    "I bought jeans last week and they don't fit. Can I send them back?",
]:
    print("-" * 58)
    print("Customer:", msg)
    print(orchestrator.orchestrate(thread_id="routing-demo", user_message=msg))
print("-" * 58)

> Want more? The repo's own [`examples/`](https://github.com/montycloud/moya/tree/main/examples) folder has `quick_start_multiagent.py`, `quick_start_flows.py`, `quick_start_subagents.py` and a full `trip_planner`, all built from these same pieces.

---
# **Key Takeaways**
---

- **ADK relies on the AI following directions**: *You tell the AI in the prompt to "summarize and stop," and you hope it obeys.*

- **MOYA uses code code as a safety net**: *It forces a hard coded rule (FunctionStep) after the AI runs, so the summary and stop action are guaranteed no matter what the AI does.*

- **ADK is faster to set up; MOYA is easier to test**: *ADK requires writing less code to get started, but MOYA lets you test your custom logic separately without running expensive LLM calls.*

- **Descriptions are infrastructure**: *In Section 2B the classifier routed on the `description=` field alone. The moment you have more than one agent, the text you wrote to be helpful to humans becomes the thing the router actually runs on.*

- **Read the framework's source, not just its README**: *MOYA's `Tool` reads parameters from `- name: desc` docstring lines; ADK reads your type hints. Same function, two schemas — and the failure is silent, not loud.*

- **Real AI safety comes from limits, not instructions**: *Prompts can fail, but an AI agent can never make an unauthorized purchase if you don't build a payment button in the first place. Giving the AI fewer abilities is far safer than asking it nicely to be careful.*


> Models are the brain, tools are the hands, and frameworks bring it all together. 
<br> From searching data with RAG to connecting systems with MCP, 
<br> Frameworks turn raw AI into a reliable assistant that plans, acts, and gets real work done safely.

### Task for you !!
Add a **fourth tool**, `remove_from_cart(item_name)`, to *either* framework (or both), and have the agent use it if `check_budget` comes back over budget. No new concepts required — one more Python function, one more line registering it.

---
# **Thank you !**
---

Author: *Aneetta Sara Shany*

Date: *2026 September 12, Saturday*